# Exercise 8 - Overfitting and Regularization
**Hardware Accelerator / computer server to use:**

$\begin{array}{lcl}
   \text{Colab/Kaggle} &:& \text{CPU} \\
  \text{CoCalc} &:& \text{Home Base} \\
 \end{array}$

**Suggested duration: ~ 30 minutes**
 
**Goals**:  <br/>
- use an existing ANN to model a non linear regression problem.
- experiment with L2 regularization as a way to reduce over-fitting when using a non-linear regression algorithm.
- investigate the impact of the training data size on regularization

### Dataset Generation

In [ ]:
import numpy as np

def generate_datasets(n_points):

    global train_x, train_y, test_x, test_y

    def generate_dataset(seed, n_points):
        train_slope  = 0.1
        train_offset = -1.0
        x = np.linspace(-10, 10, n_points).astype(np.float32)
        rng = np.random.RandomState(seed=seed)
        y = (train_slope * x + np.sin(x / 1.5) + train_offset +
                   rng.normal(0.0, 0.2, size=len(x))).astype(np.float32)
        return (x,y)

    # Training dataset
    train_x, train_y = generate_dataset(42, n_points)

    # Test dataset
    test_x, test_y = generate_dataset(43, n_points)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

generate_datasets(n_points=100)

plt.figure(1, figsize=(12, 8))
plt.plot(train_x, train_y, 'o');
plt.show()

### Data Preparation

Reshape the data into 2-D arrays as before, then normalize the data to bring it into the range [-1.0,+1.0]

In [ ]:
train_x = train_x.reshape(-1, 1)
train_y = train_y.reshape(-1, 1)
train_x = train_x - np.mean(train_x, axis=0)
train_x = train_x / np.max(train_x, axis=0)

test_x = test_x.reshape(-1, 1)
test_y = test_y.reshape(-1, 1)
test_x = test_x - np.mean(test_x, axis=0)
test_x = test_x / np.max(test_x, axis=0)

### Neural Network Model

Build a Keras model with multiple hidden layers. Note the use of the Keras `EarlyStopping` callback, which stops the run when the loss stops improving in order to reduce the run time.

In [ ]:
import tensorflow
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
import os

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

def build_and_run_regression_graph(num_layers, num_hidden, num_steps, num_runs):

    # Number of data points
    m = train_x.shape[0]

    training_loss_for_run = []
    test_loss_for_run = []
    prediction_for_run = []

    for run in range(num_runs):
        tensorflow.keras.backend.clear_session()

        model = Sequential()
        model.add(Input(shape=(1,)))

        # First hidden layers
        model.add(Dense(units=num_hidden, activation='relu'))

        # Subsequent hidden layers
        for i in range(1,num_layers):
            model.add(Dense(units=num_hidden, activation='relu'))

        # Output layer
        model.add(Dense(units=1))

        if (run == 0): model.summary()

        model.compile(loss='mean_squared_error', optimizer='adam')

        stopping = tensorflow.keras.callbacks.EarlyStopping(
            monitor='loss', min_delta=0.00001, patience=50, verbose=1)

        model.fit(train_x, train_y, epochs=num_steps, batch_size=m, verbose=0, callbacks=[stopping])

        training_loss = model.evaluate(train_x, train_y, batch_size=m, verbose=0)
        test_loss     = model.evaluate(test_x, test_y, batch_size=m, verbose=0)
        prediction    = model.predict(test_x, batch_size=m)

        print(f'Training loss = {training_loss:5.3f}, test loss = {test_loss:5.3f}')

        training_loss_for_run.append(training_loss)
        test_loss_for_run.append(test_loss)
        prediction_for_run.append(prediction)

    print(f'Training loss min/mean/max = {np.min(training_loss_for_run):5.3f}/'
          f'{np.mean(training_loss_for_run):5.3f}/{np.max(training_loss_for_run):5.3f}')

    print(f'Test     loss min/mean/max = {np.min(test_loss_for_run):5.3f}/'
          f'{np.mean(test_loss_for_run):5.3f}/{np.max(test_loss_for_run):5.3f}')

    # Plot the runs as a sanity check
    plt.figure(1, figsize=(12, 8))
    plt.plot(train_x, train_y, 'o');
    for run in range(num_runs):
        plt.plot(test_x, prediction_for_run[run])
    plt.show()

### Training  

Build and run the graph. The values of the hyperparameters have been chosen so that the model runs in a reasonable time and is able to demonstrates the effect of regularization. You may experiment with the number of layers, number of hidden units, number of steps, and number of runs if you wish.

In [ ]:
build_and_run_regression_graph(num_layers=4, num_hidden=16, num_steps=3000, num_runs=10)

### Use of L2 regularization

**Question 1**: 
Now add L2 regularization to the code above, printing out the value of the cost before adding the regularization term as well as the value of the regularization term. You could either start by making a copy of the code above or you could add a new parameter Lambda to the function *build_and_run_regression_graph*. Re-run the graph and compare the results with and without regularization. Experiment with various values of Lambda.

**Note**: do not use *lambda* (lower case) as a parameter name, since it is a Python 3 reserved keyword used to create lambda functions.

In [ ]:
#

#### Solution 

If you want some help with the answer, you can look at our answer.  Copy and paste necessary sections of the code in a new cell to run the exercise.  Do not click on the cell below unless you want to see the answer we provide!

<details>
    <summary> See our answer </summary>

    import tensorflow
    from tensorflow.keras.models        import Sequential
    from tensorflow.keras.layers        import Dense, Input
    from tensorflow.keras.regularizers  import l2

    def build_and_run_regression_graph(num_layers, num_hidden, num_steps, num_runs, Lambda=0.001):

        # Number of data points
        m = train_x.shape[0]

        training_loss_for_run = []
        test_loss_for_run = []
        prediction_for_run = []

        for run in range(num_runs):
            tensorflow.keras.backend.clear_session()

            model = Sequential()
            model.add(Input(shape=(1,)))

            # First hidden layer
            model.add(Dense(units=num_hidden, activation='relu', kernel_regularizer=l2(Lambda)))

            # Subsequent hidden layers
            for i in range(1,num_layers):
                model.add(Dense(units=num_hidden, activation='relu', kernel_regularizer=l2(Lambda)))

            # Output layer
            model.add(Dense(units=1, kernel_regularizer=l2(Lambda)))

            if (run == 0): model.summary()

            model.compile(loss='mean_squared_error', optimizer='adam')

            stopping = tensorflow.keras.callbacks.EarlyStopping(
                monitor='loss', min_delta=0.00001, patience=50, verbose=1)

            n_points = train_x.shape[0]
            model.fit(train_x, train_y, epochs=num_steps, batch_size=m, verbose=0, callbacks=[stopping])

            training_loss = model.evaluate(train_x, train_y, batch_size=m, verbose=0)
            test_loss     = model.evaluate(test_x, test_y, batch_size=m, verbose=0)
            prediction    = model.predict(test_x, batch_size=m)

            print(f'Training loss = {training_loss:5.3f}, test loss = {test_loss:5.3f}')

            training_loss_for_run.append(training_loss)
            test_loss_for_run.append(test_loss)
            prediction_for_run.append(prediction)

        print(f'Training loss min/mean/max = {np.min(training_loss_for_run):5.3f}/'
              f'{np.mean(training_loss_for_run):5.3f}/{np.max(training_loss_for_run):5.3f}')

        print(f'Test     loss min/mean/max = {np.min(test_loss_for_run):5.3f}/'
              f'{np.mean(test_loss_for_run):5.3f}/{np.max(test_loss_for_run):5.3f}')

        # Plot the runs as a sanity check
        plt.figure(1, figsize=(12, 8))
        plt.plot(train_x, train_y, 'o');
        for run in range(num_runs):
            plt.plot(test_x, prediction_for_run[run])
        plt.show()
    
	build_and_run_regression_graph(num_layers=4, num_hidden=16, num_steps=3000, num_runs=10)
</details>

### Training Dataset Size

**Question 2:** 
Repeat the run with more data samples just to demonstrate that the apparent overfitting in the previous run is an effect of the small size of the training dataset. A training dataset of just 200 points is insufficient to represent the underlying distribution, and so, unsurprisingly, the model performs worse on the test dataset. We now increase the number of data samples to roughly twice the number of trainable parameters so that the training dataset is a good representation of the underlying distribution.

In [ ]:
generate_datasets(n_points=2000)

plt.figure(1, figsize=(12, 8))
plt.plot(train_x, train_y, 'o');
plt.show()

In [ ]:
build_and_run_regression_graph(num_layers=4, num_hidden=16, num_steps=3000, num_runs=10)